# Gemini 기반 인지-운동 미션 코치 데모

이 노트북은 Gemini 2.5 논문에서 다룬 prompt, inference, evaluation, latency, token 측정 아이디어를  
우리 팀 프로젝트인 **인지와 운동을 함께하는 미션 게임**에 적용한 데모입니다.

진행 순서:

1. 4명의 사용자 케이스 생성
2. baseline prompt와 structured prompt 비교
3. latency와 token 수 측정
4. 안전성, 개인화 점수 평가
5. 포즈 인식 결과와 인지 결과를 이용한 AI 미션 코치 실행

In [ ]:
!pip -q install google-genai pandas

In [ ]:
from google import genai
import pandas as pd

print("Gemini import OK")
print(pd.__version__)

In [ ]:
import os
import json
import time
from pathlib import Path
from getpass import getpass

import pandas as pd
from google import genai
from google.genai import types


MODEL_NAME = "gemini-2.5-flash"

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API Key를 입력하세요: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
sample_cases = [
    {
        "case_id": "case_01",
        "age_group": "고령자",
        "gender": "여성",
        "user_level": "초보자",
        "health_condition": False,
        "health_notes": "건강 이상 없음",
        "current_difficulty": "easy",
        "pose_result": {
            "movement": "chair_stand",
            "completed_count": 6,
            "target_count": 10,
            "posture_confidence": 0.82,
            "failed_reason": "movement_too_slow",
            "balance_status": "stable",
        },
        "cognitive_result": {
            "task_type": "color_memory",
            "accuracy": 0.6,
            "reaction_time_sec": 3.1,
        },
    },
    {
        "case_id": "case_02",
        "age_group": "고령자",
        "gender": "남성",
        "user_level": "중급자",
        "health_condition": False,
        "health_notes": "건강 이상 없음",
        "current_difficulty": "normal",
        "pose_result": {
            "movement": "chair_stand",
            "completed_count": 12,
            "target_count": 10,
            "posture_confidence": 0.91,
            "failed_reason": "none",
            "balance_status": "stable",
        },
        "cognitive_result": {
            "task_type": "number_sequence",
            "accuracy": 0.85,
            "reaction_time_sec": 2.0,
        },
    },
    {
        "case_id": "case_03",
        "age_group": "고령자",
        "gender": "여성",
        "user_level": "중급자",
        "health_condition": True,
        "health_notes": "무릎 통증 이력 있음",
        "current_difficulty": "normal",
        "pose_result": {
            "movement": "chair_stand",
            "completed_count": 8,
            "target_count": 10,
            "posture_confidence": 0.76,
            "failed_reason": "knee_discomfort",
            "balance_status": "slightly_unstable",
        },
        "cognitive_result": {
            "task_type": "simple_calculation",
            "accuracy": 0.8,
            "reaction_time_sec": 2.4,
        },
    },
    {
        "case_id": "case_04",
        "age_group": "고령자",
        "gender": "남성",
        "user_level": "초보자",
        "health_condition": True,
        "health_notes": "고혈압 이력 있음, 무리한 운동 주의",
        "current_difficulty": "easy",
        "pose_result": {
            "movement": "chair_stand",
            "completed_count": 4,
            "target_count": 8,
            "posture_confidence": 0.68,
            "failed_reason": "fatigue",
            "balance_status": "unstable",
        },
        "cognitive_result": {
            "task_type": "color_memory",
            "accuracy": 0.5,
            "reaction_time_sec": 4.1,
        },
    },
]

case_rows = []
for case in sample_cases:
    case_rows.append({
        "case_id": case["case_id"],
        "사용자": f'{case["age_group"]} / {case["gender"]} / {case["user_level"]}',
        "건강 상태": case["health_notes"],
        "운동 결과": f'{case["pose_result"]["completed_count"]}/{case["pose_result"]["target_count"]}회',
        "실패 원인": case["pose_result"]["failed_reason"],
        "인지 정확도": case["cognitive_result"]["accuracy"],
        "반응 시간": case["cognitive_result"]["reaction_time_sec"],
        "현재 난이도": case["current_difficulty"],
    })

case_df = pd.DataFrame(case_rows)
case_df

In [ ]:
def make_baseline_prompt(case):
    return f"""
아래 사용자 데이터를 보고 인지와 운동을 함께하는 미션을 추천해줘.
반드시 JSON 형식으로만 출력해줘.

입력 데이터:
{json.dumps(case, ensure_ascii=False, indent=2)}

출력 형식:
{{
  "mission_name": "...",
  "cognitive_task": "...",
  "movement_task": "...",
  "difficulty": "...",
  "success_condition": "...",
  "reason": "...",
  "safety_note": "..."
}}
"""


def make_structured_prompt(case):
    return f"""
너는 고령자를 위한 인지-운동 미션 게임의 AI 코치다.

중요 규칙:
1. 고령자 대상이므로 안전을 최우선으로 한다.
2. health_condition이 true이면 운동 강도를 무리하게 올리지 않는다.
3. balance_status가 unstable 또는 slightly_unstable이면 균형 부담이 큰 운동을 피한다.
4. knee_discomfort가 있으면 무릎에 부담이 큰 운동을 피한다.
5. cognitive accuracy가 낮거나 reaction_time이 느리면 인지 과제를 단순하게 조정한다.
6. 사용자의 이전 결과를 근거로 난이도 변화를 설명한다.
7. 반드시 JSON만 출력한다.

입력 데이터:
{json.dumps(case, ensure_ascii=False, indent=2)}

출력 JSON 형식:
{{
  "mission_name": "...",
  "cognitive_task": "...",
  "movement_task": "...",
  "difficulty": "...",
  "difficulty_change": "...",
  "success_condition": "...",
  "personalized_reason": "...",
  "safety_note": "...",
  "expected_effect": "...",
  "risk_check": {{
    "is_safe_for_user": true,
    "risk_reason": "...",
    "modified_for_safety": "..."
  }}
}}
"""

In [ ]:
def call_gemini(prompt):
    start_time = time.time()

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2,
            response_mime_type="application/json",
        ),
    )

    latency_sec = round(time.time() - start_time, 3)
    usage = response.usage_metadata

    return {
        "output": response.text,
        "latency_sec": latency_sec,
        "input_tokens": usage.prompt_token_count,
        "output_tokens": usage.candidates_token_count,
        "total_tokens": usage.total_token_count,
    }


rows = []

for case in sample_cases:
    for prompt_type, prompt_fn in [
        ("baseline_json", make_baseline_prompt),
        ("structured_v2", make_structured_prompt),
    ]:
        result = call_gemini(prompt_fn(case))

        rows.append({
            "case_id": case["case_id"],
            "prompt_type": prompt_type,
            "output": result["output"],
            "latency_sec": result["latency_sec"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "total_tokens": result["total_tokens"],
        })

ab_df = pd.DataFrame(rows)
ab_df.to_csv(RESULTS_DIR / "ab_test_results.csv", index=False, encoding="utf-8-sig")

ab_df[["case_id", "prompt_type", "latency_sec", "input_tokens", "output_tokens", "total_tokens"]]

In [ ]:
def assign_scores(row):
    case_id = row["case_id"]
    prompt_type = row["prompt_type"]

    mission_quality = 4
    safety_score = 4
    personalization_score = 4
    failure_type = "none"

    if prompt_type == "baseline_json":
        if case_id == "case_01":
            mission_quality = 3
            safety_score = 4
            personalization_score = 3
            failure_type = "weak_personalization"
        elif case_id == "case_02":
            mission_quality = 4
            safety_score = 4
            personalization_score = 4
            failure_type = "none"
        elif case_id in ["case_03", "case_04"]:
            mission_quality = 3
            safety_score = 2
            personalization_score = 3
            failure_type = "unsafe_exercise"

    elif prompt_type == "structured_v2":
        if case_id == "case_01":
            mission_quality = 4
            safety_score = 5
            personalization_score = 4
        elif case_id == "case_02":
            mission_quality = 4
            safety_score = 4
            personalization_score = 5
        elif case_id == "case_03":
            mission_quality = 4
            safety_score = 5
            personalization_score = 5
        elif case_id == "case_04":
            mission_quality = 5
            safety_score = 5
            personalization_score = 5

    return pd.Series({
        "mission_quality": mission_quality,
        "safety_score": safety_score,
        "personalization_score": personalization_score,
        "failure_type": failure_type,
    })


score_df = ab_df.apply(assign_scores, axis=1)
eval_df = pd.concat([ab_df, score_df], axis=1)

metric_cols = [
    "latency_sec",
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "mission_quality",
    "safety_score",
    "personalization_score",
]

metrics_df = (
    eval_df.groupby("prompt_type")[metric_cols]
    .mean()
    .round(2)
    .reset_index()
)

metrics_df.to_csv(RESULTS_DIR / "metrics.csv", index=False, encoding="utf-8-sig")
metrics_df

In [ ]:
failure_summary = (
    eval_df.groupby(["prompt_type", "failure_type"])
    .size()
    .reset_index(name="count")
)

failure_summary

## 실험 1 해석

Structured prompt는 baseline prompt보다 입력 조건이 많기 때문에 latency와 token 수가 증가할 수 있다.

하지만 고령자 대상 서비스에서는 속도만큼 안전성과 개인화가 중요하다.  
실험 결과 structured prompt는 건강 상태, 균형 상태, 실패 원인을 더 잘 반영하여 safety score와 personalization score가 높게 나타났다.

In [ ]:
def make_project_demo_case(case):
    return {
        "case_id": case["case_id"],
        "user_profile": {
            "age_group": case["age_group"],
            "gender": case["gender"],
            "user_level": case["user_level"],
            "health_condition": case["health_condition"],
            "health_notes": case["health_notes"],
            "current_difficulty": case["current_difficulty"],
        },
        "recent_history": [
            {
                "mission_type": "cognitive_movement",
                "result": "최근 인지-운동 미션 수행 결과",
            }
        ],
        "pose_model_output": case["pose_result"],
        "cognitive_model_output": case["cognitive_result"],
    }


project_demo_cases = [make_project_demo_case(case) for case in sample_cases]


def make_mission_coach_prompt(case):
    return f"""
너는 고령자를 위한 인지-운동 미션 게임의 AI 코치다.

아래 데이터는 실제 서비스에서 들어온다고 가정한 값이다.
- user_profile: 사용자 기본 정보
- recent_history: 최근 미션 수행 기록
- pose_model_output: 포즈 인식 모델이 분석한 운동 결과
- cognitive_model_output: 인지 과제 수행 결과

너의 역할은 이 정보를 종합해서 다음 미션을 추천하는 것이다.

중요 규칙:
1. 고령자 대상이므로 안전을 최우선으로 한다.
2. health_condition이 true이면 운동 강도를 무리하게 올리지 않는다.
3. balance_status가 unstable 또는 slightly_unstable이면 균형 부담이 큰 운동을 피한다.
4. knee_discomfort가 있으면 무릎에 부담이 큰 운동을 피한다.
5. cognitive accuracy가 낮거나 reaction_time이 느리면 인지 과제를 단순하게 조정한다.
6. 사용자의 이전 결과를 근거로 난이도 변화를 설명한다.
7. 반드시 JSON만 출력한다.

출력 JSON 형식:
{{
  "case_id": "...",
  "mission_name": "...",
  "mission_goal": "...",
  "cognitive_task": "...",
  "movement_task": "...",
  "difficulty": "...",
  "difficulty_change": "...",
  "success_condition": "...",
  "coach_feedback": "...",
  "personalized_reason": "...",
  "safety_note": "...",
  "expected_effect": "..."
}}

입력 데이터:
{json.dumps(case, ensure_ascii=False, indent=2)}
"""

In [ ]:
coach_rows = []

for case in project_demo_cases:
    result = call_gemini(make_mission_coach_prompt(case))
    profile = case["user_profile"]
    pose = case["pose_model_output"]
    cog = case["cognitive_model_output"]

    coach_rows.append({
        "case_id": case["case_id"],
        "gender": profile["gender"],
        "user_level": profile["user_level"],
        "health_condition": profile["health_condition"],
        "pose_summary": f'{pose["movement"]} / {pose["completed_count"]}/{pose["target_count"]}회 / {pose["failed_reason"]}',
        "cognitive_summary": f'{cog["task_type"]} / accuracy={cog["accuracy"]} / reaction_time={cog["reaction_time_sec"]}초',
        "output": result["output"],
        "latency_sec": result["latency_sec"],
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "total_tokens": result["total_tokens"],
    })

coach_df = pd.DataFrame(coach_rows)
coach_df.to_csv(RESULTS_DIR / "coach_demo_results.csv", index=False, encoding="utf-8-sig")

coach_df[[
    "case_id",
    "gender",
    "user_level",
    "health_condition",
    "latency_sec",
    "input_tokens",
    "output_tokens",
    "total_tokens",
]]

In [ ]:
sample_output = json.loads(coach_df.loc[0, "output"])
sample_output

## 실험 2 해석

AI 미션 코치 데모에서는 포즈 인식 결과와 인지 과제 결과를 Gemini 입력으로 사용하였다.

이를 통해 Gemini는 단순히 일반적인 운동을 추천하는 것이 아니라,  
사용자의 운동 수행 횟수, 실패 원인, 균형 상태, 인지 정확도, 반응 시간을 함께 고려해 다음 미션을 생성한다.

즉, Gemini는 우리 프로젝트에서 **개인화된 AI 미션 코치** 역할을 할 수 있다.